In [0]:
import requests
from pyspark.sql.types import * 
from pyspark.sql.functions import current_date, split , col , lit
import json
import pandas as pd


import requests

all_data = []
page = 1
limit = 10

while True:
    skip = (page - 1) * limit
    url = f"https://dummyjson.com/users?limit={limit}&skip={skip}"
    response = requests.get(url)
    data = response.json()
    
    users = data.get("users", [])
    
    if not users:
        break
    
    all_data.extend(users)
    page += 1

print(f"Total records fetched: {len(all_data)}")


# Convert through pandas to preserve nested structures
pandas_df = pd.DataFrame(all_data)
df = spark.createDataFrame(pandas_df)

# Cast numeric columns to proper types
df = df.withColumn("id", col("id").cast("int")) \
       .withColumn("age", col("age").cast("int"))

# Flatten nested fields
flat_df = df.select(
    col("id"),
    col("firstName"),
    col("lastName"),
    col("email"),
    col("age"),
    col("gender"),
    col("phone"),
    col("username"),
    
    # Flatten nested fields
    col("address.city").alias("city"),
    col("address.state").alias("state"),
    col("company.name").alias("company_name")
)

flat_df = flat_df.withColumn("site_address", split("email", "@")[1])
flat_df = flat_df.withColumn("load_date", current_date())

display(flat_df)

db_name = "site_info"
table_name = "person_info"

# Create database
spark.sql(f"CREATE DATABASE IF NOT EXISTS {db_name}")

# Write DataFrame
flat_df.write \
    .format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable(f"{db_name}.{table_name}")